In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(PROJECT_ROOT))
# from src.utils import file_io
# from src.config import settings

In [2]:
import re
import unicodedata
import pandas as pd
import numpy as np
pd.options.display.float_format = '{:,.2f}'.format

In [3]:
def limpar_nome_pdf(nome: str) -> str:
    """
    Remove uma ou mais ocorrências de '.pdf' no final do nome (case-insensitive).
    """
    return re.sub(r"(\.pdf)+$", "", nome, flags=re.IGNORECASE)


def listar_pdfs_limpos(root: Path) -> set[str]:
    return {
        limpar_nome_pdf(p.name).lower()
        for p in root.rglob("*.pdf")
    }

def norm_nome(x: str) -> str:
    s = str(x)

    # normalização unicode e espaços estranhos
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u00A0", " ")
    s = s.strip()

    # se vier caminho completo, pega só o nome
    s = Path(s).name

    # remove qualquer combinação terminal de .pdf e/ou .md
    s = re.sub(r"(\.(pdf|md))+$", "", s, flags=re.IGNORECASE)

    # normaliza espaços
    s = re.sub(r"\s+", " ", s)

    return s.lower()

def listar_mds(root: Path) -> set[str]:
    return {
        norm_nome(p.name)
        for p in root.rglob("*.md")
    }

In [4]:
def listar_pdfs_por_ente(root: Path) -> pd.DataFrame:
    """
    Percorre uma pasta raiz onde cada subpasta representa um ente federativo
    e retorna um DataFrame com:
    - ente_federativo
    - nome_arquivo_pdf
    """

    registros = []

    for pasta_ente in root.iterdir():
        if not pasta_ente.is_dir():
            continue

        ente_federativo = pasta_ente.name

        for pdf in pasta_ente.glob("*.pdf"):
            registros.append({
                "ente_federativo": ente_federativo,
                "nome_arquivo_pdf": pdf.name
            })

    return pd.DataFrame(registros)


def moeda_br_para_float(serie: pd.Series) -> pd.Series:
    def converter(valor):
        if pd.isna(valor):
            return np.nan

        # Se já for número, retorna direto
        if isinstance(valor, (int, float)):
            return float(valor)

        valor_str = str(valor).strip()

        if valor_str == "":
            return np.nan

        # Remove símbolos (R$, espaços etc.)
        valor_str = re.sub(r"[^\d,\.]", "", valor_str)

        # Caso típico BR: tem vírgula decimal
        if "," in valor_str:
            valor_str = valor_str.replace(".", "")
            valor_str = valor_str.replace(",", ".")
            return float(valor_str)

        # Caso sem vírgula:
        # assume que já está no padrão correto
        return float(valor_str)

    return serie.apply(converter)

def padronizar_percentual(serie: pd.Series) -> pd.Series:
    def converter(valor):
        if pd.isna(valor):
            return np.nan

        valor_str = str(valor).strip()

        # Caso com %
        if "%" in valor_str:
            try:
                return float(valor_str.replace("%", "").replace(",", ".")) / 100
            except ValueError:
                return np.nan

        # Caso numérico
        try:
            num = float(valor_str.replace(",", "."))
        except ValueError:
            return np.nan

        # Se for maior que 1, interpretamos como percentual inteiro
        if num > 1:
            return num / 100

        # Caso contrário, já é fração
        return num

    return serie.apply(converter).astype(float)

In [5]:
# FILE_PATH = '../data/processed/tabela_processada_18_12_2025_11_40.parquet'
FILE_PATH = '../data/processed/Ações Afirmativas na Política Nacional Aldir Blanc - Análise de Descumprimentos.xlsx'


In [8]:
df = pd.read_excel(FILE_PATH, sheet_name='Resultado Final')

In [10]:
df.columns

Index(['ente_federativo', 'nome_pdf_pk', 'Conferência/Parecer',
       'perc_cotas_negras', 'perc_cotas_indigenas', 'perc_cotas_pcd',
       'vagas_totais', 'valor_total', 'is_novo', 'tipo_ente',
       'vagas_cotas_negras', 'vagas_cotas_indigenas', 'vagas_cotas_pcd',
       'valor_por_vaga', 'valor_cotas_negras', 'valor_cotas_indigenas',
       'valor_cotas_pcd', 'flag_cotas_negras', 'flag_cotas_indigenas',
       'flag_cotas_pcd', 'tipo_edital'],
      dtype='object')

In [132]:
def calcula_cotas_vagas_e_valores(
    df: pd.DataFrame,
    col_valor_total: str = 'valor_total',
    col_vagas_totais: str = 'vagas_totais'
) -> pd.DataFrame:

    cotas = {
        'negras': 'perc_cotas_negras',
        'indigenas': 'perc_cotas_indigenas',
        'pcd': 'perc_cotas_pcd'
    }

    # calcula vagas
    for grupo, col_perc in cotas.items():
        df[f'vagas_cotas_{grupo}'] = (
            np.floor(df[col_perc] * df[col_vagas_totais])
            .astype('Int64')
        )

    # valor unitário
    df['valor_por_vaga'] = df[col_valor_total] / df[col_vagas_totais]

    # calcula valores
    for grupo in cotas.keys():
        df[f'valor_cotas_{grupo}'] = (
            df['valor_por_vaga'] * df[f'vagas_cotas_{grupo}']
        )

    return df

def cria_flags_cotas(df: pd.DataFrame) -> pd.DataFrame:
    regras = {
        'negras': ('perc_cotas_negras', 0.25),
        'indigenas': ('perc_cotas_indigenas', 0.10),
        'pcd': ('perc_cotas_pcd', 0.05),
    }

    for grupo, (col, limite) in regras.items():
        df[f'flag_cotas_{grupo}'] = df[col] >= limite

    return df

import pandas as pd
from unicodedata import normalize

def cria_tipo_edital(
    df: pd.DataFrame,
    col_origem: str = 'nome_pdf_pk',
    col_destino: str = 'tipo_edital'
) -> pd.DataFrame:

    def normaliza_texto(s: str) -> str:
        s = str(s)
        s = normalize('NFD', s)
        s = s.encode('ascii', 'ignore').decode('utf-8')
        return s.lower()

    texto = df[col_origem].apply(normaliza_texto)

    sep = r'(^|[_\-\s\.])'
    end = r'([_\-\s\.]|$)'

    regras = [
        # CULTURA VIVA (prioridade máxima)
        (
            rf'{sep}(pncv|cultura{sep}?viva|ponto(s)?|pontao(s)?|pontoe?s?){end}',
            'CULTURA VIVA'
        ),

        # BOLSA
        (
            rf'{sep}bolsa(s)?{end}',
            'BOLSA'
        ),

        # SUBSÍDIO (com erro comum)
        (
            rf'{sep}(subsidio|subisidio)(s)?{end}',
            'SUBSÍDIO'
        ),

        # PRÊMIO / PREMIAÇÃO
        (
            rf'{sep}premi(o|os|acao|acoes)?{end}',
            'PRÊMIO'
        ),

        # FOMENTO (inclui QUALIFICAÇÃO)
        (
            rf'{sep}(fomento(s)?|qualificacao){end}',
            'FOMENTO'
        ),
    ]

    df[col_destino] = pd.NA

    for pattern, label in regras:
        mask = texto.str.contains(pattern, regex=True)
        df.loc[mask & df[col_destino].isna(), col_destino] = label

    return df




In [133]:
df = cria_flags_cotas(df)
df = cria_tipo_edital(df)

C:\Users\gabiru\AppData\Local\Temp\ipykernel_17548\3186266486.py:98: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = texto.str.contains(pattern, regex=True)
C:\Users\gabiru\AppData\Local\Temp\ipykernel_17548\3186266486.py:98: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = texto.str.contains(pattern, regex=True)
C:\Users\gabiru\AppData\Local\Temp\ipykernel_17548\3186266486.py:98: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = texto.str.contains(pattern, regex=True)
C:\Users\gabiru\AppData\Local\Temp\ipykernel_17548\3186266486.py:98: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = texto.str.contains(pattern, regex=Tr

In [134]:
df['tipo_edital'].value_counts(dropna=False)

tipo_edital
FOMENTO         279
CULTURA VIVA    100
PRÊMIO           69
SUBSÍDIO         34
BOLSA            15
<NA>              1
Name: count, dtype: int64